# LPJ-GUESS → CLM PFT conversion

This notebook defines, documents, validates, and exports the PFT correspondence used to translate LPJ-GUESS natural PFT groups into CLM natural-PFT indices.

It contains **only the conversion scheme**. It does not load FPC fields, regrid data, modify `PCT_NAT_PFT`, or create a surface dataset.

The exported CSV is deliberately machine-readable: the CLM numeric index and abbreviation are stored in separate columns, and mapped, bare-ground, and excluded PFTs have explicit treatments.

## Mapping decisions

- Trees are matched primarily by growth form, leaf form, and phenology.
- `BNE` and `BINE` both map to boreal needleleaf evergreen trees because CLM does not distinguish their shade-tolerance strategies.
- `IBS` maps to boreal broadleaf deciduous trees for this >45°N application.
- High, low, and prostrate dwarf shrubs map to CLM's single boreal shrub PFT, `BoBDS`. This necessarily maps evergreen shrubs to a deciduous CLM shrub because no evergreen boreal shrub PFT exists.
- `C3G` and `GRT` map to arctic C3 grass.
- LPJ-GUESS `CLM` (cushion forb, lichen, and moss) maps to bare ground because no corresponding natural PFT is used here.
- Peatland PFTs are recorded but excluded from this conversion.
- CLM natural-PFT index 13 (`C3`) is not modified by this scheme.

## 1. Define the PFT conversion

In [1]:
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("PFT_conversion")
CSV_PATH = OUTPUT_DIR / "lpjguess_to_clm_pft_conversion.csv"

#----------------------------------------------------------------
LPJ_NATURAL = [
    "BNE", "BINE", "BNS", "TeNE", "TeBE", "IBS", "TeBS", "C3G",
    "HSE", "HSS", "LSE", "LSS", "GRT", "EPDS", "SPDS", "CLM",
]

LPJ_PEATLAND = ["pLSE", "pLSS", "pCLM", "WetGRS", "pmoss", "C3G_wet", "C4G_wet"]

LPJ_NAMES = {
    "BNE": "Boreal needle-leaved evergreen tree",
    "BINE": "Boreal shade-intolerant needle-leaved evergreen tree",
    "BNS": "Boreal needle-leaved summergreen tree",
    "TeNE": "Temperate needle-leaved evergreen tree",
    "TeBE": "Temperate broad-leaved evergreen tree",
    "IBS": "Shade-intolerant broad-leaved summergreen tree",
    "TeBS": "Temperate broad-leaved summergreen tree",
    "C3G": "Cold C3 grass",
    "HSE": "High shrub evergreen",
    "HSS": "High shrub summergreen",
    "LSE": "Low shrub evergreen",
    "LSS": "Low shrub summergreen",
    "GRT": "Graminoid and forb tundra",
    "EPDS": "Evergreen prostrate dwarf shrub (needleleaved)",
    "SPDS": "Summergreen prostrate dwarf shrub (broadleaved)",
    "CLM": "Cushion forb, lichen and moss",
    "pLSE": "Peatland low shrub evergreen",
    "pLSS": "Peatland low shrub summergreen",
    "pCLM": "Peatland cushion/forb",
    "WetGRS": "Flood-tolerant grass/graminoid",
    "pmoss": "Peatland moss",
    "C3G_wet": "Peatland C3 grass",
    "C4G_wet": "Peatland C4 grass",
}

VEGETATION_GROUP = {
    **{pft: "tree" for pft in ["BNE", "BINE", "BNS", "TeNE", "TeBE", "IBS", "TeBS"]},
    **{pft: "grass" for pft in ["C3G", "GRT"]},
    **{pft: "shrub" for pft in ["HSE", "HSS", "LSE", "LSS", "EPDS", "SPDS"]},
    "CLM": "non-vascular",
    **{pft: "peatland" for pft in LPJ_PEATLAND},
}

CLM_NAMES = {
    0: "BG", 1: "TeNET", 2: "BoNET", 3: "BoNDT", 4: "TrBET",
    5: "TeBET", 6: "TrBDT", 7: "TeBDT", 8: "BoBDT",
    9: "TeBES", 10: "TeBDS", 11: "BoBDS", 12: "arcticC3",
    13: "C3", 14: "C4",
}

# CLM natpft index -> LPJ-GUESS PFTs summed into that target.
CONVERSION_SCHEME = {
    1: ["TeNE"],
    2: ["BNE", "BINE"],
    3: ["BNS"],
    5: ["TeBE"],
    7: ["TeBS"],
    8: ["IBS"],
    11: ["HSE", "HSS", "LSE", "LSS", "EPDS", "SPDS"],
    12: ["C3G", "GRT"],
}

BARE = 0

## 3. Validate the declared mapping

In [2]:
mapped_natural = [pft for pfts in CONVERSION_SCHEME.values() for pft in pfts]
expected_mapped = set(LPJ_NATURAL) - {"CLM"}

assert len(mapped_natural) == len(set(mapped_natural)), "An LPJ PFT is mapped more than once."
assert set(mapped_natural) == expected_mapped, (
    "Natural-PFT coverage mismatch: "
    f"missing={sorted(expected_mapped - set(mapped_natural))}, "
    f"unexpected={sorted(set(mapped_natural) - expected_mapped)}"
)
assert set(CONVERSION_SCHEME).issubset(CLM_NAMES), "Unknown CLM natpft index."
assert 13 not in CONVERSION_SCHEME, "CLM C3 (natpft 13) should remain untouched."

print(f"Validated {len(mapped_natural)} mapped natural PFTs.")
print("Validated CLM -> bare-ground treatment separately.")
print(f"Recorded {len(LPJ_PEATLAND)} excluded peatland PFTs.")

Validated 15 mapped natural PFTs.
Validated CLM -> bare-ground treatment separately.
Recorded 7 excluded peatland PFTs.


## 4. Build and export the reusable table

In [3]:
lpj_to_clm_id = {
    lpj_pft: clm_id
    for clm_id, lpj_pfts in CONVERSION_SCHEME.items()
    for lpj_pft in lpj_pfts
}
lpj_to_clm_id["CLM"] = BARE

rows = []
all_lpj_pfts = LPJ_NATURAL + LPJ_PEATLAND

for lpj_index, lpj_abbrev in enumerate(all_lpj_pfts):
    clm_id = lpj_to_clm_id.get(lpj_abbrev)

    if lpj_abbrev == "CLM":
        treatment = "mapped_to_bare"
        note = "No corresponding natural CLM PFT is used in this conversion."
    elif clm_id is not None:
        treatment = "mapped"
        note = ""
    else:
        treatment = "excluded"
        note = "Peatland PFT; excluded from the natural-PFT conversion."

    rows.append({
        "lpj_index": lpj_index,
        "lpj_abbrev": lpj_abbrev,
        "lpj_name": LPJ_NAMES[lpj_abbrev],
        "vegetation_group": VEGETATION_GROUP[lpj_abbrev],
        "clm_natpft": clm_id,
        "clm_abbrev": CLM_NAMES.get(clm_id),
        "treatment": treatment,
        "note": note,
    })

conversion_table = pd.DataFrame(rows)
conversion_table["clm_natpft"] = conversion_table["clm_natpft"].astype("Int64")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
conversion_table.to_csv(CSV_PATH, index=False)

print(f"Saved: {CSV_PATH}")
conversion_table

Saved: PFT_conversion/lpjguess_to_clm_pft_conversion.csv


,lpj_index,lpj_abbrev,lpj_name,vegetation_group,clm_natpft,clm_abbrev,treatment,note
0,0,BNE,Boreal needle-leaved evergreen tree,tree,2,BoNET,mapped,
1,1,BINE,Boreal shade-intolerant needle-leaved evergree...,tree,2,BoNET,mapped,
2,2,BNS,Boreal needle-leaved summergreen tree,tree,3,BoNDT,mapped,
3,3,TeNE,Temperate needle-leaved evergreen tree,tree,1,TeNET,mapped,
4,4,TeBE,Temperate broad-leaved evergreen tree,tree,5,TeBET,mapped,
5,5,IBS,Shade-intolerant broad-leaved summergreen tree,tree,8,BoBDT,mapped,
6,6,TeBS,Temperate broad-leaved summergreen tree,tree,7,TeBDT,mapped,
7,7,C3G,Cold C3 grass,grass,12,arcticC3,mapped,
8,8,HSE,High shrub evergreen,shrub,11,BoBDS,mapped,
9,9,HSS,High shrub summergreen,shrub,11,BoBDS,mapped,


## 5. Reload and verify reusability

In [4]:
reloaded = pd.read_csv(CSV_PATH)
mapped = reloaded.loc[reloaded["treatment"].eq("mapped")].copy()
mapped["clm_natpft"] = mapped["clm_natpft"].astype(int)

reconstructed_scheme = (
    mapped.groupby("clm_natpft", sort=True)["lpj_abbrev"]
    .apply(list)
    .to_dict()
)

assert reconstructed_scheme == CONVERSION_SCHEME
print("Reload check passed. Reconstructed conversion scheme:")
reconstructed_scheme

Reload check passed. Reconstructed conversion scheme:


{1: ['TeNE'],
 2: ['BNE', 'BINE'],
 3: ['BNS'],
 5: ['TeBE'],
 7: ['TeBS'],
 8: ['IBS'],
 11: ['HSE', 'HSS', 'LSE', 'LSS', 'EPDS', 'SPDS'],
 12: ['C3G', 'GRT']}

## Output

Running the notebook creates:

`PFT_conversion/lpjguess_to_clm_pft_conversion.csv`

Later workflows can read this CSV and reconstruct `CONVERSION_SCHEME` without copying the mapping manually.